## ***A Data-Driven Analysis of Tourist Satisfaction in SriLankan Destinations Using Online Review Analytics***

# Import Libraries

In [1]:
# ==============================
# Core Libraries
# ==============================
import pandas as pd
import numpy as np

# ==============================
# Visualization
# ==============================
import matplotlib.pyplot as plt
import seaborn as sns

# ==============================
# Text Processing
# ==============================
import random
import re
import string

# ==============================
# Natural Language Processing
# ==============================
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# ==============================
# Feature Extraction
# ==============================
from sklearn.feature_extraction.text import CountVectorizer

# ==============================
# Sentiment Analysis
# ==============================
from nltk.sentiment import SentimentIntensityAnalyzer

# ==============================
# Topic Modeling
# ==============================
from gensim import corpora
from gensim.models import LdaModel

# ==============================
# Word Cloud
# ==============================
from wordcloud import WordCloud

# ==============================
# Display Settings
# ==============================
pd.set_option('display.max_columns', None)
sns.set_style("whitegrid")

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('vader_lexicon')
nltk.download('punkt_tab') # Added to resolve the LookupError

# Reproducibility
random.seed(42)
np.random.seed(42)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\abaiy\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\abaiy\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\abaiy\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\abaiy\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


# Basic Data Inspection and Pre processing

In [ ]:
df = pd.read_csv("Reviews.csv", encoding='latin1')
df.head()

,Location_Name,Located_City,Location,Location_Type,User_ID,User_Location,User_Locale,User_Contributions,Travel_Date,Published_Date,Rating,Helpful_Votes,Title,Text
0,Arugam Bay,Arugam Bay,"Arugam Bay, Eastern Province",Beaches,User 1,"Dunsborough, Australia",en_US,8,2019-07,2019-07-31T07:53:21-04:00,5,1,Best nail spa in Arugam bay on the water!,I had a manicure here and it really was profes...
1,Arugam Bay,Arugam Bay,"Arugam Bay, Eastern Province",Beaches,User 2,"Bendigo, Australia",en_US,4,2019-06,2019-07-21T21:50:11-04:00,4,0,Best for surfing,"Overall, it is a wonderful experience. We visi..."
2,Arugam Bay,Arugam Bay,"Arugam Bay, Eastern Province",Beaches,User 3,"Melbourne, Australia",en_US,13,2019-07,2019-07-15T18:52:55-04:00,5,0,We Love Arugam Bay,"Great place to chill, swim, surf, eat, shop, h..."
3,Arugam Bay,Arugam Bay,"Arugam Bay, Eastern Province",Beaches,User 4,"Ericeira, Portugal",en_US,4,2019-06,2019-07-03T10:32:41-04:00,5,0,Sun and waves.,Good place for surf and a few stores to going ...
4,Arugam Bay,Arugam Bay,"Arugam Bay, Eastern Province",Beaches,User 5,"Pistoia, Italy",en_US,14,2019-07,2019-07-02T17:07:02-04:00,5,0,"Great swimming, surfing, great fish aznd frien...",This place is great for surfing but even if yo...


In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df["Rating"].value_counts()

In [ ]:
df["Location_Type"].value_counts()

In [ ]:
df["Located_City"].value_counts().head(10)

In [ ]:
date_columns = ["Travel_Date", "Published_Date"]

for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors="coerce", utc=True)

df.dtypes

In [ ]:
df = df.drop(columns=["User_ID"])

In [ ]:
df["review_length"] = df["Text"].apply(len)          # Creating new features
df["word_count"] = df["Text"].apply(lambda x: len(str(x).split()))
df.info()
df.describe()

# Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(6,4))           # This shows how satisfied tourists are overall.
sns.countplot(x="Rating", data=df)
plt.title("Distribution of Tourist Ratings")
plt.xlabel("Rating")
plt.ylabel("Number of Reviews")
plt.show()

In [ ]:
top_locations = df["Location_Name"].value_counts().head(10)          #This identifies popular tourist attractions.
plt.figure(figsize=(8,5))
sns.barplot(x=top_locations.values, y=top_locations.index)
plt.title("Top 10 Most Reviewed Tourist Destinations")
plt.xlabel("Number of Reviews")
plt.ylabel("Location")
plt.show()

In [ ]:
plt.figure(figsize=(7,4))             # This shows which types of attractions appear most frequently.
sns.countplot(y="Location_Type", data=df, order=df["Location_Type"].value_counts().index)
plt.title("Distribution of Attraction Types")
plt.xlabel("Number of Reviews")
plt.ylabel("Attraction Type")
plt.show()

In [ ]:
top_cities = df["Located_City"].value_counts().head(10)        # This identifies major tourism hubs.

plt.figure(figsize=(8,5))
sns.barplot(x=top_cities.values, y=top_cities.index)
plt.title("Top Cities by Number of Reviews")
plt.xlabel("Number of Reviews")
plt.ylabel("City")
plt.show()

In [ ]:
plt.figure(figsize=(6,4))           # This analyzes how detailed the reviews are.
sns.histplot(df["word_count"], bins=50)
plt.title("Distribution of Review Word Counts")
plt.xlabel("Number of Words")
plt.ylabel("Frequency")
plt.show()

In [ ]:
plt.figure(figsize=(6,4))      # This shows how often reviews are marked as helpful.
sns.histplot(df["Helpful_Votes"], bins=30)
plt.title("Distribution of Helpful Votes")
plt.xlabel("Helpful Votes")
plt.ylabel("Frequency")
plt.show()

In [ ]:
numeric_cols = ["Rating", "User_Contributions", "Helpful_Votes"]       # Check relationships between numeric variables.

corr_matrix = df[numeric_cols].corr()

plt.figure(figsize=(5,4))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm")
plt.title("Correlation Matrix")
plt.show()

# Text Pre-Processing

In [ ]:
# 6. Text Preprocessing

import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

stop_words = set(stopwords.words("english"))

def preprocess_text(text):
    if pd.isna(text):
        return ""

    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)   # remove URLs
    text = re.sub(r'[^a-zA-Z\s]', '', text)               # remove punctuation and numbers
    tokens = word_tokenize(text)                          # tokenize
    tokens = [word for word in tokens if word not in stop_words]  # remove stopwords

    return " ".join(tokens)

df["clean_text"] = df["Text"].apply(preprocess_text)

df[["Text", "clean_text"]].head()

In [ ]:
df["clean_text"].head()

In [ ]:
df["clean_word_count"] = df["clean_text"].apply(lambda x: len(x.split()))
df[["word_count", "clean_word_count"]].head()

# Word Frequency Analysis

In [ ]:
all_words = " ".join(df["clean_text"])
from collections import Counter

word_counts = Counter(all_words.split())

top_words = word_counts.most_common(20)

top_words

In [ ]:
freq_df = pd.DataFrame(top_words, columns=["Word", "Frequency"])
freq_df

In [ ]:
plt.figure(figsize=(8,5))
sns.barplot(data=freq_df, x="Frequency", y="Word")
plt.title("Top 20 Most Frequent Words in Reviews")
plt.xlabel("Frequency")
plt.ylabel("Word")
plt.show()

In [ ]:
from wordcloud import WordCloud

wordcloud = WordCloud(width=800, height=400, background_color="white").generate(all_words)

plt.figure(figsize=(10,5))
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.title("Word Cloud of Tourist Reviews")
plt.show()

# Sentiment Analysis

In [ ]:
from nltk.sentiment import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

df["sentiment_score"] = df["clean_text"].apply(lambda x: sia.polarity_scores(x)["compound"])
def get_sentiment(score):
    if score > 0.05:
        return "Positive"
    elif score < -0.05:
        return "Negative"
    else:
        return "Neutral"

df["sentiment"] = df["sentiment_score"].apply(get_sentiment)
df[["clean_text", "sentiment_score", "sentiment"]].head()

In [ ]:
plt.figure(figsize=(6,4))         # This shows the overall emotional tone of tourist experiences.
sns.countplot(x="sentiment", data=df)
plt.title("Sentiment Distribution of Tourist Reviews")
plt.xlabel("Sentiment")
plt.ylabel("Number of Reviews")
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(x="Rating", y="sentiment_score", data=df)
plt.title("Sentiment Score vs Rating")
plt.show()

# Topic Modeling (TDA)

In [ ]:
texts = [text.split() for text in df["clean_text"]]

from gensim import corpora

dictionary = corpora.Dictionary(texts)
dictionary.filter_extremes(no_below=5, no_above=0.5)
corpus = [dictionary.doc2bow(text) for text in texts]

from gensim.models import LdaModel

lda_model = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=5,
    random_state=42,
    passes=10
)
topics = lda_model.print_topics(num_words=10)

for topic in topics:
    print(topic)

In [ ]:
def get_topic(text):
    bow = dictionary.doc2bow(text.split())
    topics = lda_model.get_document_topics(bow)
    return max(topics, key=lambda x: x[1])[0]

df["topic"] = df["clean_text"].apply(get_topic)

import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(6,4))
sns.countplot(x="topic", data=df)
plt.title("Distribution of Topics in Tourist Reviews")
plt.xlabel("Topic")
plt.ylabel("Number of Reviews")
plt.show()

In [ ]:
for idx, topic in lda_model.print_topics(num_words=10):
    print(f"Topic {idx}: {topic}")

In [ ]:
topic_labels = {
    0: "Wildlife Safari Tourism",
    1: "Scenic Views and Hiking",
    2: "Religious and Cultural Sites",
    3: "Beach Tourism",
    4: "Tea Plantation Tours"
}

df["topic_name"] = df["topic"].map(topic_labels)

In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(y="topic_name", data=df, order=df["topic_name"].value_counts().index)

plt.title("Distribution of Tourism Experience Themes")
plt.xlabel("Number of Reviews")
plt.ylabel("Topic")
plt.show()

In [ ]:
sentiment_topic = df.groupby("topic_name")["sentiment_score"].mean()

sentiment_topic.plot(kind="bar", figsize=(8,5))
plt.title("Average Sentiment Score by Topic")
plt.xlabel("Tourism Theme")
plt.ylabel("Average Sentiment Score")
plt.show()

In [ ]:
topics = lda_model.show_topics(num_topics=5, num_words=10, formatted=False)

for topic_id, words in topics:
    print(f"\nTopic {topic_id}")
    print([word for word, prob in words])

In [ ]:
for topic_id, words in lda_model.show_topics(num_topics=5, num_words=20, formatted=False):

    topic_words = dict(words)

    wordcloud = WordCloud(width=600, height=400, background_color='white').generate_from_frequencies(topic_words)

    plt.figure(figsize=(6,4))
    plt.imshow(wordcloud)
    plt.axis("off")
    plt.title(f"Topic {topic_id} Word Cloud")
    plt.show()

In [ ]:
topic_rating = df.groupby("topic_name")["Rating"].mean().sort_values(ascending=False)

plt.figure(figsize=(8,5))
topic_rating.plot(kind="bar")

plt.title("Average Rating by Tourism Theme")
plt.xlabel("Tourism Theme")
plt.ylabel("Average Rating")
plt.show()

In [ ]:
topic_city = pd.crosstab(df["Located_City"], df["topic_name"])

topic_city.head()

In [ ]:
top_cities = df["Located_City"].value_counts().head(10).index

plt.figure(figsize=(10,6))
sns.countplot(data=df[df["Located_City"].isin(top_cities)],
              y="Located_City",
              hue="topic_name")

plt.title("Tourism Themes by City")
plt.xlabel("Number of Reviews")
plt.ylabel("City")
plt.show()

In [ ]:
city_sentiment = df.groupby("Located_City")["sentiment_score"].mean().sort_values(ascending=False)

city_sentiment.head(10)

In [ ]:
plt.figure(figsize=(8,5))
city_sentiment.head(10).plot(kind="bar")

plt.title("Top Cities by Average Sentiment Score")
plt.xlabel("City")
plt.ylabel("Average Sentiment Score")
plt.show()

In [ ]:
city_rating = df.groupby("Located_City")["Rating"].mean().sort_values(ascending=False)

plt.figure(figsize=(8,5))
city_rating.head(10).plot(kind="bar")

plt.title("Top Cities by Average Tourist Rating")
plt.xlabel("City")
plt.ylabel("Average Rating")
plt.show()

In [ ]:
df.to_csv("processed_tourism_reviews.csv", index=False)

In [ ]:
df.info()

In [ ]:
for value in df["Location_Name"].unique():
    print(value)

In [ ]:
for value in df["Located_City"].unique():
    print(value)

In [ ]:
# Validation-only cell: use the single canonical functions defined earlier (do not redefine them)
# This computes checks and lists remaining unmapped locations for review

# Compute check columns without redefining functions
df['province_check'] = df['Location'].apply(extract_province)
df['district_check'] = df.apply(infer_district, axis=1)

# Rows where either province or district could not be inferred
unmapped = df[df['province_check'].isnull() | df['district_check'].isnull()]

print("Unique Locations with missing province or district (examples):")
print(sorted(unmapped['Location'].dropna().unique())[:200])
print('\nCount of rows with missing province or district:', len(unmapped))

# show a few distinct examples for manual mapping decisions
display(unmapped[['Location','Located_City','province_check','district_check']].drop_duplicates().head(50))